# BSDS demo — hidden brain-state dynamics with `ffs_bsds`

This notebook fits **Bayesian Switching Dynamical Systems** (Taghia et al. 2018; Cai et al. 2024)
to ROI time series and walks through the outputs, with figures inline.

BSDS is a *switching factor-analysis* model: an HMM over a handful of discrete **brain states**,
each with its own low-rank+diagonal covariance (its **functional-connectivity** pattern), fit by
variational Bayes. It gives you, moment by moment, which latent state the brain is in, how states
transition, how long they dwell, and what network pattern defines each.

Here we use synthetic data with a known ground truth so you can see the model recover it. At the
end are notes on pointing it at real data (and the equivalent one-line `ffs_bsds` call).

In [ ]:
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt

from fastfuncstuff.dynamics.bsds.model import fit_bsds
from fastfuncstuff.dynamics.states import compute_state_stats
from fastfuncstuff.dynamics import plots

## 1. A synthetic switching dataset

Each **session** is a `(D, N)` array — `D` ROIs by `N` timepoints. We simulate `K` states, each
with a distinct mean and a low-rank+diagonal covariance, and a *sticky* transition matrix (states
persist, matching real brain dynamics). Feeding several sessions is exactly how you'd analyse a
**densely-sampled individual**: runs/sessions become the model's session list.

In [ ]:
def simulate(k=4, d=12, r=2, t=500, n_sessions=3, stay=0.96, seed=0):
    rng = np.random.default_rng(seed)
    means = rng.standard_normal((k, d)) * 4.0
    chols, covs = [], []
    for _ in range(k):
        w = rng.standard_normal((d, r)) * 0.7
        cov = w @ w.T + 0.3 * np.eye(d)
        covs.append(cov); chols.append(np.linalg.cholesky(cov))
    trans = np.full((k, k), (1 - stay) / (k - 1)); np.fill_diagonal(trans, stay)
    sessions, truth = [], []
    for _ in range(n_sessions):
        z = np.empty(t, int); z[0] = rng.integers(k)
        for i in range(1, t): z[i] = rng.choice(k, p=trans[z[i-1]])
        y = np.stack([means[z[i]] + chols[z[i]] @ rng.standard_normal(d) for i in range(t)], axis=1)
        sessions.append(torch.tensor(y, dtype=torch.float64)); truth.append(z)
    return sessions, truth, np.array(covs)

sessions, truth, true_covs = simulate()
print(f'{len(sessions)} sessions, D={sessions[0].shape[0]} ROIs, N={sessions[0].shape[1]} TRs each')

## 2. Fit BSDS

`fit_bsds` runs variational Bayes with several random restarts, keeps the best by free energy,
and decodes the most-likely state sequence. `n_states` is an **upper bound** — the ARD prior
prunes unused states/factors, so you can start generous.

In [ ]:
model = fit_bsds(sessions, n_states=4, max_ldim=4, n_init=5, n_iter=100, seed=0)
stats = compute_state_stats(model, tr=0.72)  # TR in seconds -> lifetimes in seconds
print('converged:', model.converged, 'in', len(model.objective_history), 'iters')
print('occupancy:', np.round(stats.group_occupancy, 3))

## 3. Did it fit? — free energy

The variational free energy **F** must increase monotonically. A clean rising curve that flattens
means the fit converged; a jagged or still-climbing curve means raise `n_iter` or add restarts.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
plots.plot_convergence(model, ax)
plt.show()

## 4. The core view — state probability time courses

This is the figure to read first. Each line is the posterior probability of one state over time.
Crisp, near-binary switching between well-separated states means the model found real structure;
mushy overlapping probabilities mean the states aren't well identified (ambiguous data, wrong
`n_states`, or too little data).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3))
plots.plot_state_timecourses(model, run_idx=0, ax=ax, tr=0.72)
plt.show()

The MAP (Viterbi) path collapses that to a single most-likely state per frame — a compact ribbon:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 1.1))
plots.plot_state_ribbon(model, run_idx=0, ax=ax, tr=0.72)
plt.show()

## 5. Switching dynamics — the transition matrix

Row *i*, column *j* is P(state *j* next | state *i* now). Brain states are **sticky**, so a good
fit has a dominant diagonal — here it should recover the ~0.96 stay probability we simulated.

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
plots.plot_transition_matrix(model, ax)
plt.show()

## 6. How much, how long — occupancy & mean lifetime

Fractional **occupancy** is the share of time in each state; **mean lifetime** is the average
dwell duration (in seconds here, via the TR). The gray dots on occupancy are per-session values,
showing across-session variability. These per-state summaries are what the papers relate to
behaviour (e.g. occupancy of the optimal state predicts task performance).

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(9, 3.2))
plots.plot_occupancy(stats, a0)
plots.plot_lifetime(stats, a1)
plt.tight_layout(); plt.show()

## 7. What defines each state — activation & functional connectivity

Each state has a **mean** (activation profile across ROIs) and a **covariance**. The covariance,
normalised to a correlation, is the state's **dynamic functional connectivity** — the network
pattern that state represents. This is the payoff of the factor-analysis observation model:
clean, denoised per-state FC.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
plots.plot_state_means(model, ax)
plt.show()

In [ ]:
plots.plot_state_fc(stats)
plt.show()

## 8. Everything at once — the QC panel

`qc_report` bundles all of the above into one figure — the same PNG `ffs_bsds` writes next to its
outputs. Glance at it to answer *did it fit, do the states make sense?*

In [ ]:
from IPython.display import Image
qc_path = '/tmp/bsds_demo_qc.png'
plots.qc_report(model, stats, qc_path, tr=0.72)
Image(qc_path)

## 9. Matching states across fits

Separate fits label their states arbitrarily. `match_states` aligns them — either by **state-space
closeness** (symmetric KL between the per-state Gaussians) or **temporal closeness** (correlation
of the probability time courses). This is how Cai et al. tracked one shared state across seven
different tasks.

In [ ]:
from fastfuncstuff.dynamics.matching import match_states
model_b = fit_bsds(sessions, n_states=4, max_ldim=4, n_init=5, n_iter=100, seed=7)
m = match_states(model, model_b, method='state_space')
print('fit-A state -> fit-B state:', dict(zip(m.row_ind.tolist(), m.col_ind.tolist())))

## 10. The CEBRA bridge

BSDS (discrete states, temporal structure) and [CEBRA](https://cebra.ai) (continuous nonlinear
manifold) decompose *orthogonal axes* of the same dynamics. `export.py` prepares the hand-off
(no CEBRA dependency): concatenate sessions into the `(N, D)` matrix CEBRA wants, and grab the
frame-aligned state labels to **colour a CEBRA embedding**. If the states occupy distinct
territories on the manifold, that's convergent evidence they're real.

In [ ]:
from fastfuncstuff.dynamics.export import prepare_cebra_inputs, frame_aligned_labels
X, lengths = prepare_cebra_inputs(sessions)
labels, _ = frame_aligned_labels(model)
print('CEBRA input:', X.shape, '| labels:', labels.shape, '| session lengths:', lengths)
print('# then: emb = CEBRA().fit_transform(X); colour emb by `labels`;')
print('#       state_embedding_separation(emb, labels) scores how distinct they are.')

## Using real data

Replace `simulate()` with your own sessions — each a `(D, N)` ROI time series (`torch`/`numpy`).
Get there from 4-D fMRI with `fastfuncstuff.dynamics.parcellate` (atlas, or data-driven
contiguity-constrained Ward for a precision individual), and `preprocess_sessions` for per-run
detrend + z-score. Keep `D` in the tens–low-hundreds.

The whole pipeline is also one CLI call:

```bash
ffs_bsds -input run*.1D -prefix out/sub01 -n_states 6 -tr 0.72 -plots all
# or from volumes:
ffs_bsds -input run*.nii.gz -parcellation atlas -atlas schaefer.nii.gz \
         -prefix out/sub01 -tr 0.72 -plots all
```

which writes the model bundle, per-run state time courses, and these same figures.